### DL: Python 3 with NVIDIA T4 GPU
### Other: Apple M1 Pro

In [2]:
import pickle
from pathlib import Path
from time import perf_counter
import xgboost

path = Path("../data/xgb_fold7.pkl")

print("XGBoost version:", xgboost.__version__, flush=True)
start = perf_counter()

with path.open("rb") as f:
    target_model = pickle.load(f)

print(f"load done: {perf_counter() - start:.1f}sec", flush=True)

XGBoost version: 3.2.0
load done: 0.1sec


In [3]:
import ast
import json
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem
from mordred import Calculator, descriptors

sys.path.insert(0, str(Path("../src").resolve()))
from dl import TorchMLPClassifier  # noqa: F401

BIND_MODEL = path

target_columns = [
    'AATSC2c', 'AATSC0d', 'MATS2d', 'MATS1pe', 'MATS1p', 'MATS5p',
    'MATS4i', 'MATS5i', 'GATS5c', 'GATS1dv', 'GATS4dv', 'NssNH2',
    'NdNH', 'NssssN', 'SssssN', 'IC3', 'EState_VSA9', 'n8HRing',
    'n8AHRing', 'n10FHRing', 'n10FaHRing', 'n12FAHRing',
    'JGI3', 'JGI7', 'JGI8', 'JGI9', 'JGI10', 'PetitjeanIndex'
]

/Users/idahyeon/miniforge3/envs/adrb2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
fp_columns = [f"X{i}" for i in range(1, 1025)]
target_columns = fp_columns + target_columns

needed_descriptors = set(target_columns[1024:])
calculator = Calculator(
    [d for d in Calculator(descriptors, ignore_3D=True).descriptors
     if str(d) in needed_descriptors],
    ignore_3D=True
)

In [5]:
def make_features(smiles, columns):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError("This SMILE is invalid.")

    canonical = Chem.MolToSmiles(mol, canonical=True)
    mol = Chem.MolFromSmiles(canonical)

    fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=1024)
    bits = np.zeros(1024, dtype=int)
    DataStructs.ConvertToNumpyArray(fp, bits)

    row = {f"X{i + 1}": int(bit) for i, bit in enumerate(bits)}
    row.update({
        str(desc): value
        for desc, value in zip(calculator.descriptors, calculator(mol))
    })

    X = pd.DataFrame([row]).reindex(columns=columns)
    X = X.apply(pd.to_numeric, errors="coerce")
    X = X.replace([np.inf, -np.inf], np.nan)
    return canonical, X


In [6]:
smiles = "O=c1ccc2ccc(O)c(O)c2o1" #dapnetin

canonical, X_target = make_features(smiles, target_columns)
target_probability = float(target_model.predict_proba(X_target)[0, 1])
print(f"ADRB2-TargetNet probability: {target_probability:.4f}")

[12:31:43] DEPRECATION WARNING: please use MorganGenerator


ADRB2-TargetNet probability: 0.9889
